# HydraY NNUE: training-recipe experiments (dead l1 neurons, LR tail, fine-tune)

Runtime -> **GPU A100** (High-RAM if offered), set `ARM` in the cell "configuration",
then "Run all". The training arms need about **75 GiB of free disk** (the v9 pool
decompressed); the `census` arm needs no GPU and almost no disk.

### Why this branch
Measured 2026-10-28 on the holdout (positions no net trained on, `nnue/tools/l1census.cpp`):
every deep net trained with the current recipe has about 30% of its l1 neurons
**dead** (never firing), almost all in the output buckets of 18-32 pieces, which hold
42% of the data and most of what the engine evaluates in play:

| net | dead l1 neurons per output bucket 0..7 (16 each) | total |
|---|---|---|
| 4.2.0 (shipped) | 0 0 2 2 **8 12 5 9** | 38/128 |
| 4.1.0 | 0 0 3 3 **7 10 8 10** | 41/128 |
| ab_std | 1 2 2 0 **7 10 9 7** | 38/128 |
| ab_c324 | 1 0 3 2 **4 13 9 10** | 42/128 |

Bucket 5 (22-25 pieces) of 4.2.0 evaluates with 4 neurons out of 16. A dead neuron's
l1 row is exactly zero: l1's inputs (the pairwise products) are all >= 0 and bullet
starts l1's bias at 0, so a neuron pushed below zero for every input gets no gradient
again, and AdamW's decay (0.01) then zeroes it. The feature transformer has no dead
neuron. Also, bullet's default +-1.98 weight clip binds the float tail too, which is
never quantised: 11 of the 128 l2 weights sit on it, two of the four live neurons of
bucket 5 among them. The count per bucket changes from run to run (bucket 5: 10, 12,
13), noise that every one-run-per-arm architecture A/B carried. Fixing it changes
neither the net file nor the engine: same architecture, same inference, same speed.

Second lever, independent: the LR. The 4.2.0 recipe ends at 1e-4 (StepLR x0.1 at
superbatch 80); bullet's reference recipe for this very architecture
(`examples/progression/4_multi_layer.rs`: 1024 -> 16, 8 output buckets) decays with a
cosine to 1e-3 * 0.3^5 = 2.4e-6, forty times lower. The budget curve (+29, +10, +2.7
per doubling) was only ever measured with the step schedule.

Third: HydraY always trains from scratch, while Stockfish retrains every new net from
the previous one (official-stockfish/Stockfish PRs 4314 and 4465).

### Results (SPRT, [0, 5], 2026-10-29/30)

| new vs base | TC | games | Elo | verdict |
|---|---|---|---|---|
| `ft-control` vs 4.2.0 | 4+0.04 | 1164 | +22.7 +- 12.6 | clear gain, stopped |
| `ft-revive` vs `ft-control` | 4+0.04 | 9818 | about +1 | even: reviving alone gives no Elo |
| `antidead` vs `ft-revive` | 4+0.04 | 998 | about -28 | `antidead` alone loses, stopped |
| `ft-antidead` vs `ft-revive` | 4+0.04 | 8000 | +1.8 +- 4.9 | even at the cap |
| `ft-antidead` vs `ft-revive` | 10+0.1 | 7296 | +7.3 +- 4.8 | H1 |
| `ft-antidead` vs 4.2.0 | 4+0.04 | 3290 | +15.9 +- 7.7 | H1 |

`ft-antidead` is the `antidead` net fine-tuned for 40 superbatches with its dead neurons
revived: 0/128 dead l1 neurons, and it replaced 4.2.0 in `nnue/net/hydray.nnue` on
2026-10-30. Its gain over `ft-revive` showed at 10+0.1 only, so test fine-tunes at
10+0.1 too. The closed arms (`ft-control`, `ft-revive`, `antidead` alone) are gone:
`antidead-ft` runs the winning recipe end to end.

### Arms
`ARM` selects one; everything not listed is the 4.2.0 recipe (v9 pool read as one
stream, 8 output buckets, WDL 0.3, batch 16,384, eval scale 400, squared error).

| `ARM` | superbatches | what changes | question |
|---|---|---|---|
| `census` | none, no GPU | nothing is trained | when did the 4.2.0 run's neurons die? (its checkpoints on Drive) |
| `antidead-ft` | 160 + 40 | phase `antidead`: l1 bias starts at 0.1, warmup 3000 batches, float tail unclipped; phase `ft-antidead`: starts from that net, cosine 1e-4 -> 2.5e-6, warmup 3000 batches, dead neurons revived (bias 0.1, l2 weight 0) | the recipe of the shipped net, for new data |
| `cosine` | 160 | cosine 1e-3 -> 2.4e-6 instead of StepLR | is the low-LR tail worth Elo? |
| `escale280` | 160 | eval scale 280 (the K fitted on the v8/v9 labels) | low priority |
| `pow25` | 160 | loss abs(error)^2.5, as Stockfish | low priority |

An arm is a list of phases run in order, each with its own net id
`hydray-v9-<phase>-<superbatches>sb`. A phase whose final net is already on Drive
(`<net id>.nnue`) is skipped: an `antidead` net trained earlier is reused, not trained
again (delete it from Drive to retrain). Do not bundle: each arm is tested alone, and
winners are combined later in a run of their own (bundles of untested items lost three
times).

The knobs are environment variables read by `trainer_deep.rs` (list in its header);
unset, they reproduce the 4.2.0 recipe. With `HYDRAY_INIT_NET` the trainer checks,
before the first batch, that bullet evaluates the starting net like the float
reference in `sanity_deep.rs` (under 1 cp on five positions): a layout error stops the
run there. `sanity_deep <net> --init-check` runs the same conversion on any machine.

### What to look at during a run
After every stage the notebook prints the census of the new checkpoint: `antidead`
should keep buckets 4-7 near zero, and in `ft-antidead` the revived neurons should
stay alive. Running losses compare only between arms with the same eval scale and loss.
The verdict is the SPRT (last cell).

In [ ]:
# --- helper: any failed command stops the notebook, output streams live ---
import subprocess, os, re, glob, hashlib

def sh(cmd, check=True):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    out = []
    for line in p.stdout:
        print(line, end='', flush=True)
        out.append(line)
    if p.wait() != 0 and check:
        raise RuntimeError(f'FAILED (exit {p.returncode}): {cmd}')
    return ''.join(out)

gpu = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader',
                     shell=True, capture_output=True, text=True).stdout.strip()
print('GPU:', gpu or 'none')
sh('nproc; free -g | head -2; df -h /content | tail -1')

In [ ]:
# --- configuration ---
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'

ARM    = 'census'        # census | antidead-ft | cosine | escale280 | pow25
STAGE  = 20              # checkpoint to Drive every STAGE superbatches (multiple of 10)
BRANCH = 'nnue-recipe'
BUCKETS = 8              # the 4.2.0 layout

REPO    = '/content/hydray'
TRAINER = f'{REPO}/nnue/trainer'
LOCAL   = '/content/data'
INIT_NET = f'{REPO}/nnue/net/hydray.nnue'          # the shipped net (ft-antidead), checked by sha256 below
INIT_SHA = '8c71f3d779202e2e48634843b52ff20e20b193e3c0b16d74e452693a530d9b7e'

ANTIDEAD = {'HYDRAY_L1B_INIT': '0.1', 'HYDRAY_WARMUP': '3000', 'HYDRAY_FLOAT_CLIP': '64'}
FT = {'HYDRAY_LR': 'cosine', 'HYDRAY_LR_START': '0.0001', 'HYDRAY_LR_FINAL': '0.0000025',
      'HYDRAY_WARMUP': '3000', 'HYDRAY_FLOAT_CLIP': '64', 'HYDRAY_REVIVE': '1', 'HYDRAY_L1B_INIT': '0.1'}
ARMS = {  # name: phases run in order, each (id, superbatches, environment of the trainer)
    'census':      [],
    'antidead-ft': [('antidead',    160, ANTIDEAD),
                    ('ft-antidead', 40,  dict(FT, HYDRAY_INIT_NET=f'{DRIVE}/hydray-v9-antidead-160sb.nnue'))],
    'cosine':      [('cosine',      160, {'HYDRAY_LR': 'cosine'})],
    'escale280':   [('escale280',   160, {'HYDRAY_EVAL_SCALE': '280'})],
    'pow25':       [('pow25',       160, {'HYDRAY_LOSS': 'pow2.5'})],
}
assert ARM in ARMS, f'unknown ARM {ARM}'
PHASES = ARMS[ARM]

def use_phase(i):
    """Makes phase i current: the stage helpers read these globals."""
    global PHASE, TOTAL_SB, ENV, NET_ID, ENV_PREFIX
    PHASE, TOTAL_SB, ENV = PHASES[i]
    NET_ID = f'hydray-v9-{PHASE}-{TOTAL_SB}sb'
    ENV_PREFIX = ' '.join(f'{k}={v}' for k, v in ENV.items())

if ARM != 'census':
    assert gpu, 'no GPU: change the runtime type (only ARM = "census" runs without one)'
    assert STAGE % 10 == 0 and all(sb % STAGE == 0 for _, sb, _ in PHASES), \
        'stage ends must be checkpoints (save_rate 10)'

# The v9 pool (MANIFEST.txt of nnue/data/v9pack). The record counts catch a truncated
# upload or a partial decompression, which otherwise end without an error.
EXPECTED_RECORDS = {
    'hydray_v9_part1.bin.zst': 647_094_461,
    'hydray_v9_part2.bin.zst': 647_094_461,
    'hydray_v9_part3.bin.zst': 647_094_463,
}
NET_FILE_BYTES = 6_425_664
TRAINER_LINE   = f'trainer_deep: 1024 -> 16 -> 1, {BUCKETS} output buckets'

def find(name):
    hits = glob.glob(f'{DRIVE}/**/{name}', recursive=True)
    assert hits, f'{name} not found on Drive'
    return hits[0]

PARTS = {name: find(name) for name in EXPECTED_RECORDS}
for name, path in PARTS.items():
    print(f'{name}: {os.path.getsize(path)/2**30:6.2f} GiB  {path}')
print(f'\narm {ARM}:')
for i in range(len(PHASES)):
    use_phase(i)
    print(f'  phase {PHASE}: {TOTAL_SB} superbatches, net id {NET_ID}, environment {ENV}')

In [ ]:
# --- Rust, zstd, clone, checks, build ---
sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
sh('apt-get -qq install -y zstd >/dev/null')
sh(f'rm -rf {REPO} && git clone --depth 1 --branch {BRANCH} https://github.com/ThomasGhione/HydraY {REPO}')
sh(f'cd {REPO} && git log --oneline -1')

# Not ceremonial: a past run trained for an hour an architecture other than the one
# believed, because the clone was wrong and a checkpoint name says nothing about its contents.
tr = open(f'{TRAINER}/src/bin/trainer_deep.rs').read()
sd = open(f'{TRAINER}/src/bin/sanity_deep.rs').read()
assert 'const HIDDEN_SIZE: usize = 1024;' in tr and 'const L1_SIZE: usize = 16;' in tr
assert 'HYDRAY_INIT_NET' in tr and 'HYDRAY_FLOAT_CLIP' in tr, 'trainer_deep.rs has no recipe knobs: wrong branch'
assert 'pub fn to_bullet' in sd and 'pub fn dead_rows' in sd
assert hashlib.sha256(open(INIT_NET, 'rb').read()).hexdigest() == INIT_SHA, 'nnue/net/hydray.nnue is not the shipped net (ft-antidead)'

CARGO = f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH env CUDA_PATH=/usr/local/cuda HYDRAY_OUTPUT_BUCKETS={BUCKETS}'
if ARM != 'census':
    sh(f'{CARGO} cargo build -r --bin trainer_deep --features cuda')
sh(f'{CARGO} cargo build -r --bin sanity_deep')
sh(f'{TRAINER}/target/release/sanity_deep {INIT_NET} --init-check')

# The activation census (C++, the engine's own NNUE code). If this compiler cannot
# build it, the census falls back to the all-zero rows that sanity_deep counts.
L1CENSUS = '/content/l1census'
for std in ('c++23', 'c++2b'):
    sh(f'cd {REPO} && g++ -std={std} -O2 -march=native nnue/tools/l1census.cpp nnue/nnue_deep.cpp -o {L1CENSUS}', check=False)
    if os.path.exists(L1CENSUS):
        break
print('l1census:', 'built' if os.path.exists(L1CENSUS) else 'NOT built, zero rows only')

In [ ]:
# --- data: the three parts decompressed once (census: a 600k-record sample only) ---
# One zstd process per part: the Drive mount caches what a process reads and frees it
# when the process exits, so the peak is the raw data plus one part's cache.
os.makedirs(LOCAL, exist_ok=True)
SAMPLE = f'{LOCAL}/sample.bin'
if ARM == 'census':
    if not (os.path.exists(SAMPLE) and os.path.getsize(SAMPLE) == 32 * 600_000):
        sh(f'zstd -dc --long=27 "{PARTS["hydray_v9_part3.bin.zst"]}" | head -c {32 * 600_000} > {SAMPLE}')
    assert os.path.getsize(SAMPLE) == 32 * 600_000
else:
    need = sum(32 * n for n in EXPECTED_RECORDS.values())
    have = sum(os.path.getsize(f'{LOCAL}/{n[:-4]}') for n in EXPECTED_RECORDS if os.path.exists(f'{LOCAL}/{n[:-4]}'))
    st = os.statvfs('/content'); free = st.f_bavail * st.f_frsize
    print(f'free {free/2**30:.1f} GiB, still to write {(need - have)/2**30:.1f} GiB')
    assert free > (need - have) + 14 * 2**30, 'not enough disk: use an A100 runtime (or one with more disk)'
    DATA_FILES = []
    for name, path in PARTS.items():
        raw = f'{LOCAL}/{name[:-4]}'                 # strip .zst
        want = 32 * EXPECTED_RECORDS[name]
        if not (os.path.exists(raw) and os.path.getsize(raw) == want):
            # --long=27: the parts are compressed with a 128 MiB window.
            sh(f'zstd -d -T0 --long=27 -f -c "{path}" > {raw}')
        got = os.path.getsize(raw)
        assert got == want, f'{name}: {got} bytes after decompression, expected {want}'
        print(f'{name}: {got // 32:,} records, size checked', flush=True)
        DATA_FILES.append(raw)
    DATA = ','.join(DATA_FILES)
    SAMPLE = DATA_FILES[-1]                          # the census reads its first 600k records
    total = need // 32
    sbs = sum(sb for _, sb, _ in PHASES)
    print(f'\ntraining stream: {total/1e9:.3f}B records, {sbs} superbatches = {sbs*6104*16384/total:.2f} epochs')

In [ ]:
# --- stage helpers and census (RUN THIS ALSO WHEN RESTARTING) ---

def census(qbin):
    """Dead l1 neurons per output bucket: all-zero rows, and the activation census."""
    sh(f'{TRAINER}/target/release/sanity_deep {qbin} | grep "dead l1"')
    if os.path.exists(L1CENSUS):
        sh(f'{L1CENSUS} {qbin} {SAMPLE} 0 600000')

def stage_cmd(end, start, resume_from):
    # STAGE_END and the recipe variables must sit right in front of cargo (after `env`):
    # `STAGE_END=40 cd dir && cargo ...` gives the variable to `cd` only.
    args = f'{DATA} {TOTAL_SB} {NET_ID}'
    if resume_from is not None:
        args += f' {start} checkpoints/{NET_ID}-{resume_from}'
    return f'{CARGO} {ENV_PREFIX} STAGE_END={end} cargo run -r --bin trainer_deep --features cuda -- {args}'

def check_output(out, first_stage):
    assert TRAINER_LINE in out, f'the trainer did not print "{TRAINER_LINE}": wrong build'
    recipe = re.search(r'^recipe: .*$', out, re.M)
    assert recipe, 'no recipe line: wrong branch'
    for v in ENV.values():
        assert v in recipe.group(0), f'recipe line misses {v}: {recipe.group(0)}'
    assert ('revive true' in recipe.group(0)) == ('HYDRAY_REVIVE' in ENV)
    if first_stage and 'HYDRAY_INIT_NET' in ENV:
        assert out.count('init check ') == 5, 'the start check did not run'

def check_checkpoint(end):
    q = f'{TRAINER}/checkpoints/{NET_ID}-{end}/quantised.bin'
    assert os.path.exists(q), f'missing {q}'
    assert os.path.getsize(q) == NET_FILE_BYTES, f'{q}: {os.path.getsize(q)} bytes, not the 8-bucket layout'
    return q

def save_to_drive(end):
    # Copies the checkpoint to Drive and CHECKS it arrived: the Drive mount writes
    # through a cache, so an unfinished upload would pass for a finished one.
    ck  = f'{TRAINER}/checkpoints/{NET_ID}-{end}'
    dst = f'{DRIVE}/{NET_ID}-{end}'
    sh(f'rm -rf "{dst}" && cp -r {ck} {DRIVE}/')
    size = lambda p: sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(p) for f in fs)
    assert os.path.isdir(dst) and size(dst) == size(ck), f'incomplete copy on Drive: {dst}'
    print(f'superbatch {end} on Drive ({size(dst)/2**20:.0f} MiB, checked)', flush=True)

def run_stages(done=0):
    assert done % STAGE == 0, f'{done} is not a stage boundary'
    prev = done if done else None
    for k in range(done // STAGE, TOTAL_SB // STAGE):
        end, start = (k + 1) * STAGE, k * STAGE + 1
        print(f'\n===== {PHASE}, stage {k+1}/{TOTAL_SB//STAGE}: superbatches {start}-{end} =====', flush=True)
        out = sh(stage_cmd(end, start, prev))
        check_output(out, prev is None)
        census(check_checkpoint(end))
        save_to_drive(end)
        prev = end

def finish():
    """Final checks of the current phase, and its net on Drive (the next phase starts from it)."""
    final = check_checkpoint(TOTAL_SB)
    sh(f'{TRAINER}/target/release/sanity_deep {final}')
    census(final)
    sh(f'cp {final} "{DRIVE}/{NET_ID}.nnue" && cmp {final} "{DRIVE}/{NET_ID}.nnue"')
    print(f'\nnet on Drive: {DRIVE}/{NET_ID}.nnue')
    print('Sanity: startpos small, the mirrored pair (wN e4 / bN e5) identical, a queen up ~ +1000 or more.')

def run_arm(first=0, done=0):
    """Runs the phases from `first` on; `done` superbatches of phase `first` are already trained."""
    for i in range(first, len(PHASES)):
        use_phase(i)
        if done == 0 and os.path.exists(f'{DRIVE}/{NET_ID}.nnue'):
            print(f'{NET_ID}.nnue already on Drive: phase {PHASE} skipped', flush=True)
            continue
        run_stages(done=done)
        finish()
        done = 0


In [ ]:
# --- ARM = 'census': when did the 4.2.0 run's neurons die? ---
# The 4.2.0 net was trained by colab_ob16.ipynb with BUCKETS = 8, so its checkpoints
# are the Drive folders hydray-v9-ob8-160sb-<superbatch>. Set CENSUS_RUN for another run.
CENSUS_RUN = 'hydray-v9-ob8-160sb'
if ARM != 'census':
    print('not the census arm: nothing to do')
else:
    census(INIT_NET)
    dirs = sorted(glob.glob(f'{DRIVE}/{CENSUS_RUN}-*/quantised.bin'),
                  key=lambda p: int(p.split('-')[-1].split('/')[0]))
    print(f'{len(dirs)} checkpoints of {CENSUS_RUN} on Drive')
    for q in dirs:
        census(q)
    print('\nNeurons dead by superbatch 10-20 -> the start of training kills them (bias, warmup).')
    print('Dying through the 80 high-LR superbatches -> the LR matters more than the start.')

In [ ]:
# --- training ---
# Do NOT run this cell when restarting: use the cell "RESTART" below.
if ARM == 'census':
    print('census arm: nothing to train')
else:
    run_arm()

In [ ]:
# --- the arm's nets on Drive ---
# Each phase is checked and copied to Drive by run_arm (finish), the last one included.
for i in range(len(PHASES)):
    use_phase(i)
    net = f'{DRIVE}/{NET_ID}.nnue'
    print(f'{net}: {"on Drive" if os.path.exists(net) else "MISSING"}')

In [ ]:
# --- RESTART (ONLY if the session died mid-run) ---
#   1. run the cells from "helper" to "stage helpers" (the data cell reuses the parts
#      already on local disk, or decompresses them again in a new session);
#   2. do NOT run the training cell;
#   3. set RESTART = True, PHASE_ID = the phase that was running and DONE = its last
#      superbatch saved on Drive. The phases after it then run as usual.
# With RESTART = False this cell does nothing, so "Run all" is safe.
RESTART  = False
PHASE_ID = 'antidead'
DONE     = 20

if not RESTART:
    print('restart off (RESTART = False): nothing to do')
else:
    first = [p[0] for p in PHASES].index(PHASE_ID)
    use_phase(first)
    ck = f'{DRIVE}/{NET_ID}-{DONE}'
    assert os.path.isdir(ck), f'checkpoint not found on Drive: {ck}'
    os.makedirs(f'{TRAINER}/checkpoints', exist_ok=True)
    sh(f'cp -r "{ck}" {TRAINER}/checkpoints/')
    check_checkpoint(DONE)
    run_arm(first, DONE)

## After a run, on the laptop (branch `nnue-recipe` or `dev`: the engine is the same)

Net on Drive: `hydray-v9-<phase>-<superbatches>sb.nnue`, the last phase's is the arm's net
(`antidead-ft`: `hydray-v9-ft-antidead-40sb.nnue`).

**1. Reorder for sparsity** (same function, faster l1; the shipped net is reordered,
a fresh one is not, and a fine-tune moves the sparsity too: reorder every candidate):

    g++ -std=c++23 -O2 -march=native nnue/tools/reorder.cpp nnue/nnue_deep.cpp -o /tmp/reorder
    zstd -dc --long=27 hydray_v9_part3.bin.zst | head -c 640000 > /tmp/sample.bin
    ./chess datagen-dump /tmp/sample.bin 20000 | sed -n 's/^[0-9]*: \([^|]*\) |.*/\1/p' > /tmp/positions.fen
    /tmp/reorder hydray-v9-<ARM>-<N>sb.nnue /tmp/positions.fen <ARM>.nnue

**2. Checks**: `./chess nnue-selftest <ARM>.nnue` prints "nnue-selftest OK".

**3. Census and holdout loss** on the holdout (the first 20,000,400 records of
`v8_std.t0.bin` are outside the pool):

    g++ -std=c++23 -O2 -march=native nnue/tools/l1census.cpp nnue/nnue_deep.cpp -o /tmp/l1census
    /tmp/l1census <ARM>.nnue nnue/data/v8_std.t0.bin
    g++ -std=c++23 -O2 -march=native nnue/tools/holdoutloss.cpp nnue/nnue_deep.cpp -o /tmp/holdoutloss
    /tmp/holdoutloss <ARM>.nnue nnue/data/v8_std.t0.bin --count 20000000

Compare per bucket with the same command on `nnue/net/hydray.nnue` (`escale280`: add
`--scale 280`, and its loss is not comparable with the others).

**4. SPRT** against the shipped net, one binary, both nets through `EvalFile`: 4+0.04
and 10+0.1, Threads=1, [0, 5], 8000-game cap (accept at the cap only with LOS >= 95%). One SPRT at a time, and only
when no other job loads the laptop (a datagen timing session may be running).

    share/scripts/sprt.sh start <ARM>-vs-shipped ./chess ./chess \
        NEW_OPTS="EvalFile=$PWD/<ARM>.nnue" BASE_OPTS="EvalFile=$PWD/nnue/net/hydray.nnue" CPUS=4-11 CONC=8

For `antidead-ft`, test the final `ft-antidead` net only: the `antidead` net alone lost
about 28 Elo. Do not SPRT a checkpoint taken before the end of its LR schedule.